# LUMO Inference Demo

Interactive inference-time notebook for the trained support agent.

This demo shows:
- Natural-language agent and customer turns
- Per-turn environment variables and metrics
- Live turn-by-turn playback
- Batch evaluation and transcript export

Default checkpoint: `Simulation_4/artifacts/phase10_prod/models/best_model.zip`


In [2]:
!pip install ipywidgets

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]


## 1. Project Paths, Imports, and Runtime Checks

In [4]:
import json
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import gymnasium as gym
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display
from stable_baselines3 import PPO

# Resolve repo root BEFORE importing project modules.
cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "Simulation_4").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise RuntimeError(
        "Could not find repository root containing Simulation_4. "
        "Open this notebook from inside the project workspace."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from Simulation_4.env.agent_response_generator import AgentResponseGenerator
from Simulation_4.env.support_env import SupportEnv
from Simulation_4.training.nlp_observation import NLPObservationWrapper
from Simulation_4.training.text_observation import TextOnlyObservationWrapper

artifacts_root = repo_root / "Simulation_4" / "artifacts"
checkpoint_candidates = [
    artifacts_root / "phase10_prod" / "models" / "best_model.zip",
    artifacts_root / "phase10_prod" / "models" / "final_model.zip",
    artifacts_root / "phase10_final" / "models" / "best_model.zip",
    artifacts_root / "phase10_final" / "models" / "final_model.zip",
]

print(f"Repo root: {repo_root}")
print(f"Artifacts root: {artifacts_root}")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"ipywidgets available: {widgets is not None}")
print("Candidate checkpoints:")
for path in checkpoint_candidates:
    print(f"  - {path} {'(exists)' if path.exists() else '(missing)'}")

Repo root: /Users/tishabhavsar/RL_Project /AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service
Artifacts root: /Users/tishabhavsar/RL_Project /AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service/Simulation_4/artifacts
Python: 3.14.0
NumPy: 2.4.3
Pandas: 3.0.1
ipywidgets available: True
Candidate checkpoints:
  - /Users/tishabhavsar/RL_Project /AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service/Simulation_4/artifacts/phase10_prod/models/best_model.zip (exists)
  - /Users/tishabhavsar/RL_Project /AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service/Simulation_4/artifacts/phase10_prod/models/final_model.zip (exists)
  - /Users/tishabhavsar/RL_Project /AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service/Simulation_4/artifacts/phase10_final/models/best_model.zip (exists)
  - /Users/tishabhavsar/RL_Project /AdaptiveBandit--C

## 2. Load Trained PPO Checkpoint and Build Inference Environment

In [5]:
MODEL_PATH = next((p for p in checkpoint_candidates if p.exists()), None)
if MODEL_PATH is None:
    raise FileNotFoundError("No PPO checkpoint found in phase10_prod/phase10_final.")

ACTION_NAMES = {
    0: "AskInfo",
    1: "ProvideSolution",
    2: "AffectiveRepair",
    3: "Escalate",
    4: "Close",
}

DEFAULT_SEED = 90000
DEFAULT_MAX_TURNS = 20
DEFAULT_DETERMINISTIC = True

print(f"Loading PPO model from: {MODEL_PATH}")
model = PPO.load(str(MODEL_PATH), device="cpu")
model_obs_dim = int(np.prod(model.observation_space.shape))
print(f"Model observation space: {model.observation_space} ({model_obs_dim} dims)")
print(f"Action space: {model.action_space}")


class PadObservationWrapper(gym.Wrapper):
    def __init__(self, env: gym.Env, target_dim: int, agent_response_generator: AgentResponseGenerator | None = None):
        super().__init__(env)
        self.target_dim = int(target_dim)
        self.agent_response_generator = agent_response_generator
        self.observation_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(self.target_dim,), dtype=np.float32)

    @property
    def state(self) -> dict[str, Any]:
        return getattr(self.env, "state", {})

    @property
    def ACTION_NAMES(self) -> dict[int, str]:
        return getattr(self.env, "ACTION_NAMES", ACTION_NAMES)

    def _pad(self, obs: np.ndarray) -> np.ndarray:
        obs = np.asarray(obs, dtype=np.float32).reshape(-1)
        padded = np.zeros(self.target_dim, dtype=np.float32)
        limit = min(len(obs), self.target_dim)
        padded[:limit] = obs[:limit]
        return padded

    def reset(self, *, seed: int | None = None, options: dict | None = None):
        obs, info = self.env.reset(seed=seed, options=options)
        return self._pad(obs), info

    def step(self, action: int):
        action_name = self.ACTION_NAMES.get(int(action), f"Action{int(action)}")
        agent_text = None
        if self.agent_response_generator is not None:
            try:
                history = list(getattr(self.env, "conversation_history", []))
                agent_text = self.agent_response_generator.generate(
                    action_name=action_name,
                    conversation_history=history,
                    policy_context=str(getattr(self.env, "rag_context", {}).get("policy_context", "")),
                    classification=None,
                )
            except Exception:
                agent_text = None
        obs, reward, done, truncated, info = self.env.step(int(action), agent_text=agent_text)
        return self._pad(obs), reward, done, truncated, info


def build_inference_env():
    base_env = SupportEnv(artifacts_root=str(artifacts_root), nlg_enabled=True)
    if model_obs_dim == 9:
        wrapped_env = NLPObservationWrapper(base_env, agent_response_generator=AgentResponseGenerator())
        wrapper_name = "NLPObservationWrapper"
    elif model_obs_dim == 386:
        wrapped_env = TextOnlyObservationWrapper(base_env)
        wrapper_name = "TextOnlyObservationWrapper"
    else:
        wrapped_env = PadObservationWrapper(base_env, target_dim=model_obs_dim, agent_response_generator=AgentResponseGenerator())
        wrapper_name = f"PadObservationWrapper({model_obs_dim})"
    return wrapped_env, wrapper_name

inference_env, wrapper_name = build_inference_env()
print(f"Chosen inference wrapper: {wrapper_name}")
print(f"Wrapped observation space: {inference_env.observation_space}")
print(f"Wrapped action space: {inference_env.action_space}")


@dataclass
class TurnRecord:
    turn: int
    action_id: int
    action: str
    reward: float
    cumulative_reward: float
    terminal_type: str
    turn_count: int
    progress: float
    information: float
    frustration: float
    failed_streak: int
    resolved: int
    escalated: int
    dropped_off: int
    tier: str
    persona: str
    lumo_label: str
    display_name: str
    agent_text: str
    customer_text: str
    transition: dict[str, Any] = field(default_factory=dict)


print("Model and environment ready.")

Loading PPO model from: /Users/tishabhavsar/RL_Project /AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service/Simulation_4/artifacts/phase10_prod/models/best_model.zip
Model observation space: Box(0.0, 1.0, (9,), float32) (9 dims)
Action space: Discrete(5)
DocumentStore loaded:
  lumo_doc1_company_product_overview.md: 22 chunks
  lumo_doc2_support_policies.md: 48 chunks
  lumo_doc3_issue_playbooks.md: 59 chunks
  Total chunks: 129
  Mean chunk size: 787 characters
  Min chunk size: 45 characters
  Max chunk size: 1000 characters


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chosen inference wrapper: NLPObservationWrapper
Wrapped observation space: Box(0.0, 1.0, (9,), float32)
Wrapped action space: Discrete(5)
Model and environment ready.


## 3. Natural-Language Rendering Layer for State and Agent Action

In [ ]:
def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except Exception:
        return float(default)


def extract_latest_messages(history: list[dict[str, str]]) -> tuple[str, str]:
    agent_text = ""
    customer_text = ""
    if not isinstance(history, list):
        return agent_text, customer_text
    for message in reversed(history):
        if not isinstance(message, dict):
            continue
        role = str(message.get("role", "")).strip()
        content = str(message.get("content", "")).strip()
        if not content:
            continue
        if not customer_text and role == "assistant":
            customer_text = content
        elif not agent_text and role == "user":
            agent_text = content
        if agent_text and customer_text:
            break
    return agent_text, customer_text


def state_snapshot(state: dict[str, Any], info: dict[str, Any]) -> dict[str, Any]:
    transition = info.get("last_transition_outcome", {}) if isinstance(info, dict) else {}
    history = info.get("conversation_history", []) if isinstance(info, dict) else []
    agent_text, customer_text = extract_latest_messages(history)
    return {
        "turn_count": int(state.get("turn_count", 0)),
        "progress": round(safe_float(state.get("progress", 0.0)), 4),
        "information": round(safe_float(state.get("information", 0.0)), 4),
        "frustration": round(safe_float(state.get("frustration", 0.0)), 4),
        "failed_streak": int(state.get("failed_streak", 0)),
        "resolved": int(state.get("resolved", 0)),
        "escalated": int(state.get("escalated", 0)),
        "dropped_off": int(state.get("dropped_off", 0)),
        "tier": str(state.get("tier", "unknown")),
        "persona": str(state.get("persona_label", "unknown")),
        "lumo_label": str(state.get("lumo_label", state.get("display_name", "unknown"))),
        "display_name": str(state.get("display_name", state.get("subflow", "unknown"))),
        "terminal_type": str(transition.get("terminal_type", "")),
        "outcome": str(transition.get("outcome", "")),
        "agent_text": agent_text,
        "customer_text": customer_text,
        "slots_revealed": transition.get("slots_revealed", []),
        "terminal_reward": round(safe_float(transition.get("terminal_reward", 0.0)), 4),
        "per_turn_reward": round(safe_float(transition.get("per_turn_reward", 0.0)), 4),
    }


def transcript_html(history: list[dict[str, str]], max_messages: int = 12) -> str:
    items = []
    if not isinstance(history, list):
        history = []
    for message in history[-max_messages:]:
        role = str(message.get("role", "")).strip().title() or "Unknown"
        content = str(message.get("content", "")).strip()
        color = "#0f766e" if role.lower() == "user" else "#7c2d12"
        if role.lower() == "assistant":
            role = "Customer"
        elif role.lower() == "user":
            role = "Agent"
        items.append(
            f'<div style="margin:0.25rem 0; padding:0.4rem 0.6rem; border-left:4px solid {color}; background:#fafafa;">'
            f'<div style="font-size:0.78rem; font-weight:700; color:{color}; margin-bottom:0.15rem;">{role}</div>'
            f'<div style="white-space:pre-wrap; line-height:1.35;">{content}</div>'
            f'</div>'
        )
    return "".join(items) if items else '<div style="color:#666;">No transcript yet.</div>'


def build_metrics_df(turn_log: list[TurnRecord]) -> pd.DataFrame:
    if not turn_log:
        return pd.DataFrame()
    frame = pd.DataFrame([record.__dict__ for record in turn_log])
    return frame


print("State rendering helpers ready.")

## 4. Step-Level Inference Function and Live Episode Runner

In [ ]:
class DemoEpisode:
    def __init__(self, model: PPO, env: gym.Env, seed: int, deterministic: bool = True, max_turns: int = 20):
        self.model = model
        self.env = env
        self.seed = int(seed)
        self.deterministic = bool(deterministic)
        self.max_turns = int(max_turns)
        self.reset()

    def reset(self):
        self.obs, self.info = self.env.reset(seed=self.seed)
        self.done = False
        self.turn_log: list[TurnRecord] = []
        self.cumulative_reward = 0.0
        self.turn_index = 0
        self.last_state = dict(getattr(self.env, "state", {}))
        return self.obs, self.info

    def predict_action(self) -> tuple[int, float | None]:
        action, _ = self.model.predict(self.obs, deterministic=self.deterministic)
        return int(action), None

    def step(self, action_id: int | None = None) -> TurnRecord:
        if self.done:
            raise RuntimeError("Episode already finished. Reset the demo to start a new one.")
        if self.turn_index >= self.max_turns:
            self.done = True
            raise RuntimeError("Reached the maximum turn cap for this demo episode.")

        if action_id is None:
            action_id, _ = self.predict_action()
        action_id = int(action_id)
        action_name = ACTION_NAMES.get(action_id, f"Action{action_id}")

        self.obs, reward, done, truncated, info = self.env.step(action_id)
        self.cumulative_reward += float(reward)
        self.turn_index += 1
        self.last_state = dict(getattr(self.env, "state", {}))

        snapshot = state_snapshot(self.last_state, info)
        record = TurnRecord(
            turn=self.turn_index,
            action_id=action_id,
            action=action_name,
            reward=float(reward),
            cumulative_reward=float(self.cumulative_reward),
            terminal_type=str(snapshot.get("terminal_type", "")),
            turn_count=int(snapshot.get("turn_count", 0)),
            progress=float(snapshot.get("progress", 0.0)),
            information=float(snapshot.get("information", 0.0)),
            frustration=float(snapshot.get("frustration", 0.0)),
            failed_streak=int(snapshot.get("failed_streak", 0)),
            resolved=int(snapshot.get("resolved", 0)),
            escalated=int(snapshot.get("escalated", 0)),
            dropped_off=int(snapshot.get("dropped_off", 0)),
            tier=str(snapshot.get("tier", "unknown")),
            persona=str(snapshot.get("persona", "unknown")),
            lumo_label=str(snapshot.get("lumo_label", "unknown")),
            display_name=str(snapshot.get("display_name", "unknown")),
            agent_text=str(snapshot.get("agent_text", "")),
            customer_text=str(snapshot.get("customer_text", "")),
            transition=dict(info.get("last_transition_outcome", {}) or {}),
        )
        self.turn_log.append(record)
        self.info = info
        self.done = bool(done or truncated or self.turn_index >= self.max_turns)
        return record

    def run_to_end(self):
        while not self.done:
            self.step()
        return self.turn_log

    def to_frame(self) -> pd.DataFrame:
        return build_metrics_df(self.turn_log)


print("DemoEpisode class ready.")

## 5. Interactive Controls with ipywidgets